<a href="https://colab.research.google.com/github/aronwilbert/COMP8420-Group-L-Healthcare/blob/main/COMP8420_Group_L_Healthcare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# === CELL 1: FORCE SYNCHRONIZE PYTORCH AND LLM ARCHITECTURES ===
!pip install --upgrade --force-reinstall numpy pandas scipy scikit-learn textblob spacy datasets gliner transformers torch torchvision
!python -m spacy download en_core_web_md
!python -m textblob.download_corpora lite
!pip install contractions

  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached textblob-0.20.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached spacy-3.8.14-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (28 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached gliner-0.2.26-py3-none-any.whl.metadata (10 kB)
  Using cached transformers-5.9.0-py3-none-any.whl.metadata (33 kB)
  Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached python_dateutil-2.9.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 23.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
Finished.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
# === CELL 2: MODERNISED NLP EXECUTION CELL ===
import pandas as pd
from textblob import TextBlob
from sklearn.metrics.pairwise import cosine_similarity
import spacy
from gliner import GLiNER
from datasets import load_dataset
import contractions

In [2]:
# 1. Load the raw master dataset
master_dataset = load_dataset("xDAN-datasets/ChatDoctor_HealthCareMagic_112k", split="train")
print(f"📊 Initial Raw Master Dataset Samples: {len(master_dataset)}")

# --- VISUAL 1: DISPLAY RAW DATASET BEFORE STRUCTURAL FILTERING ---
print("\n--- 🛑 RAW DATASET PREVIEW (BEFORE FILTERING / RAW INGESTION) ---")
df_raw_preview = pd.DataFrame(master_dataset.select(range(5)))
display(df_raw_preview)

# 2. Filter out rows where 'input' or 'output' are missing or completely empty
clean_master = master_dataset.filter(
    lambda x: x["input"] is not None and x["output"] is not None and
              str(x["input"]).strip() != "" and str(x["output"]).strip() != ""
)
print(f"\n🧹 Structurally Valid Master Dataset Samples (After NA/Blank Removal): {len(clean_master)}")
print(f"❌ Total Defective Rows Filtered: {len(master_dataset) - len(clean_master)}")

# 3. Securely isolate EXACTLY 5,000 clean samples for the project
dataset = clean_master.select(range(5000))
print(f"✅ Final Target Samples Isolated for Analysis Pipeline: {len(dataset)}")

# --- VISUAL 2: DISPLAY SUBSET BEFORE TEXT PREPROCESSING ---
df_clean = pd.DataFrame({
    'input': [row['input'] for row in dataset],
    'output': [row['output'] for row in dataset]
})

print("\n--- 📋 VERIFIED BASELINE DATASET SNAPSHOT (PRE-TEXT CLEANING) ---")
display(df_clean.head(5))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


📊 Initial Raw Master Dataset Samples: 112165

--- 🛑 RAW DATASET PREVIEW (BEFORE FILTERING / RAW INGESTION) ---


,conversations,input,output
0,"[{'from': 'human', 'value': 'I woke up this mo...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"[{'from': 'human', 'value': 'My baby has been ...",My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"[{'from': 'human', 'value': 'Hello, My husband...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"[{'from': 'human', 'value': 'lump under left n...",lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,"[{'from': 'human', 'value': 'I have a 5 month ...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...



🧹 Structurally Valid Master Dataset Samples (After NA/Blank Removal): 112156
❌ Total Defective Rows Filtered: 9
✅ Final Target Samples Isolated for Analysis Pipeline: 5000

--- 📋 VERIFIED BASELINE DATASET SNAPSHOT (PRE-TEXT CLEANING) ---


,input,output
0,I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [27]:
import re
import pandas as pd

print("================================================================================")
print("🔬 STEP-BY-STEP FILTER: IDENTIFYING INTERNAL MID-WORD PERIODS (10 ROWS)")
print("================================================================================")

# 1. Grab our 10 raw inputs
raw_inputs_10 = df_clean['input'].astype(str).tolist()

def process_periods_only(text):
    if not isinstance(text, str):
        return ""

    # Rule 1: Remove multiple dots (e.g., "...", "..") -> replace with a single space
    text = re.sub(r'\.{2,}', ' ', text)

    # Rule 2: Remove dots at the beginning of a word (e.g., ".may" -> "may")
    text = re.sub(r'(?<=\s)\.([A-Za-z0-9])', r'\1', text)
    text = re.sub(r'^([A-Za-z0-9])', r'\1', text)

    # Rule 3: Remove dots at the end of a word (e.g., "cough." -> "cough")
    # This leaves dots that are trapped completely in the middle of characters untouched
    text = re.sub(r'([A-Za-z0-9])\.(?=\s|$)', r'\1', text)

    return text

# Run the rule set
processed_10 = [process_periods_only(t) for t in raw_inputs_10]

# 2. Extract ONLY the words where the period is still sitting in the middle
mid_word_periods = []
for text in processed_10:
    for word in text.split():
        # A mid-word period means there are letters/numbers on BOTH sides of a single dot
        if re.search(r'[A-Za-z0-9]\.[A-Za-z0-9]', word):
            mid_word_periods.append(word)

# Make the list unique
unique_mid_words = sorted(list(set(mid_word_periods)))

print("📊 ARRAYS CONTAINING PERIODS IN THE MIDDLE OF THE WORD:")
print("-" * 80)
print(unique_mid_words)
print("=" * 80)

🔬 STEP-BY-STEP FILTER: IDENTIFYING INTERNAL MID-WORD PERIODS (10 ROWS)
📊 ARRAYS CONTAINING PERIODS IN THE MIDDLE OF THE WORD:
--------------------------------------------------------------------------------
['(0.72),', '(0.99)', '(1.5', '(1.5+0+0).', '(10.4),', '(108.2),', '(16-12.5', '(2.1)', '(2.54', '(26.27)then', '(3.05).', '(3.3);', '(3.7).', '(4.1),', '(4.20),', '(4.29)', '(4.5ml,', '(7.5).', '(7.6', '(8.16)and', '(8.9*3.5cm)', '(9.1', '(@99.5)', '(T.T.P', '(e.g', '(i.e', '(i.e.,', '*5.0)', ',-normal.BUN-5,creat-normal,but', ',5.8', ',B.King', ',Kerala.Presently', ',T.Bil-', ',morphi.e', '-1.5', '-1.50', '-1.75', '-2.6', '-2.9', '-4.5', '-7.0', '-99.5', '-fsh1.4-lh', '.(1.3cm)', '/7.Do', '0.0', '0.01', '0.025%', '0.06', '0.09\\".What', '0.1%', '0.1%,', '0.1).', '0.17Serum', '0.2', '0.2-1', '0.25', '0.3,', '0.333', '0.40x0.38x0.73cm', '0.43', '0.45', '0.4mg+dutasteride', '0.5', '0.5%SLUGGISH', '0.5mg', '0.5previous', '0.6', '0.61', '0.66mg/dL,', '0.6cm', '0.6mm', '0.7', '0.8', '0.

In [34]:
import pandas as pd
import re

print("================================================================================")
print("🧼 EXECUTING FULL-SCALE CLINICAL PREPROCESSING ENGINE (5,000 ROWS)")
print("================================================================================")

def execute_final_period_logic_v3(text):
    if not isinstance(text, str):
        return ""

    # Standardize to lowercase immediately to handle varying patient typing uniformly
    text = text.lower()

    # 🔴 STEP 1: HARD LOCK KEY EXCEPTIONS FIRST (Protects them from subsequent rules)
    text = re.sub(r'i\.e\.', '_LOCK_IE_', text)
    text = re.sub(r'\bi\.e\b', '_LOCK_IE_', text)
    text = re.sub(r'e\.g\.', '_LOCK_EG_', text)
    text = re.sub(r'\be\.g\b', '_LOCK_EG_', text)
    text = re.sub(r'dr\.', '_LOCK_DR_', text)

    # STEP 2: Strip brackets completely to clean up glued metadata boundaries
    text = re.sub(r'[\(\)]', ' ', text)

    # STEP 3: Mask and protect vital clinical decimals and dates (e.g., 0.72 or 04.10.2010)
    text = re.sub(r'(\d+)\.(\d+)', r'\1_SAFE_DEC_\2', text)

    # STEP 4: Collapse multiple repeating dots (ellipses like '...' or '..') to a space
    text = re.sub(r'\.{2,}', ' ', text)

    # STEP 5: Split fused sentence text/number boundaries (e.g., "13.i" -> "13 i", "medicine.it" -> "medicine it")
    text = re.sub(r'([a-z0-9])\.([a-z])', r'\1 \2', text)

    # STEP 6: Clean up any leftover floating or trailing punctuation noise (, ! ? or terminal dots)
    text = re.sub(r'[.,;!]+(?=\s|$)', '', text)

    # 🟢 STEP 7: RESTORE ORIGINAL PROTECTED EXCEPTIONS TO DESTINATION FORMAT
    text = text.replace('_LOCK_IE_', 'i.e')
    text = text.replace('_LOCK_EG_', 'e.g')
    text = text.replace('_LOCK_DR_', 'dr.')
    text = text.replace('_SAFE_DEC_', '.')

    # STEP 8: Eliminate structural double spacing noise
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the pipeline safely to all 5,000 rows in your master DataFrame
df_clean['clean_input_periods'] = df_clean['input'].astype(str).apply(execute_final_period_logic_v3)

print("🎉 SUCCESS: All 5,000 rows processed and saved to column 'clean_input_periods'!")
print("=" * 80)
print("📊 DETAILED VERIFICATION PANEL (ROWS 0 - 49)")
print("=" * 80)

# Pull the first 50 rows to inspect the transformations
raw_inputs_50 = df_clean['input'].head(1000).astype(str).tolist()
cleaned_inputs_50 = df_clean['clean_input_periods'].head(1000).tolist()

for idx in range(1000):
    print(f"\n📍 DATASET ROW INDEX: {idx}")
    print("-" * 70)
    print(f"🔴 ORIGINAL INPUT :\n   \"{raw_inputs_50[idx]}\"")
    print(f"🟢 PERIOD CLEANED  :\n   \"{cleaned_inputs_50[idx]}\"")
    print("=" * 80)

Streaming output truncated to the last 5000 lines.

📍 DATASET ROW INDEX: 375
----------------------------------------------------------------------
🔴 ORIGINAL INPUT :
   "hello, i am a girl 14 years of age. 2 days ago i had my first panic attack. i went to my family doctor but she didnt help much. my mother often has panic attacks and she has recently been diognosed with bipoloar disorder. i went back to school today and it seemed like everything that got me upset or just generally got to me was gonna start another panic attack.. any help or advice???"
🟢 PERIOD CLEANED  :
   "hello i am a girl 14 years of age 2 days ago i had my first panic attack i went to my family doctor but she didnt help much my mother often has panic attacks and she has recently been diognosed with bipoloar disorder i went back to school today and it seemed like everything that got me upset or just generally got to me was gonna start another panic attack any help or advice???"

📍 DATASET ROW INDEX: 376
----------

In [18]:
import pandas as pd
from collections import Counter
import string

print("================================================================================")
print("🔬 MARATHON COHORT: EXTRACTING PUNCTUATED WORDS ACROSS ALL 5,000 ROWS")
print("================================================================================")

# Step 1: Ingest all 5,000 rows
all_inputs = df_clean['input'].astype(str).tolist()

# Step 2: Initialize raw collection buckets
period_words = []
comma_words = []
slash_words = []
question_words = []
other_punc_words = []

# Step 3: Scan every single word in the entire dataset
for text in all_inputs:
    for word in text.split():
        # Check for Period
        if '.' in word:
            period_words.append(word)
        # Check for Comma
        if ',' in word:
            comma_words.append(word)
        # Check for Slash
        if '/' in word:
            slash_words.append(word)
        # Check for Question Mark
        if '?' in word:
            question_words.append(word)

print("🎉 EXTRACTION COMPLETE! COLLATING UNIQUE COHORTS...")
print("=" * 80)

# Step 4: Display the unique outputs ranked by frequency
print(f"\n💬 1. ALL WORDS CONTAINING A PERIOD (.) [Unique count: {len(set(period_words))}]")
print("-" * 80)
# Extract the top 60 most common raw punctuated tokens
period_sample = [word for word, count in Counter(period_words).most_common(60)]
print(period_sample)

print(f"\n📌 2. ALL WORDS CONTAINING A COMMA (,) [Unique count: {len(set(comma_words))}]")
print("-" * 80)
comma_sample = [word for word, count in Counter(comma_words).most_common(60)]
print(comma_sample)

print(f"\n⚡ 3. ALL WORDS CONTAINING A SLASH (/) [Unique count: {len(set(slash_words))}]")
print("-" * 80)
slash_sample = [word for word, count in Counter(slash_words).most_common(60)]
print(slash_sample)

print(f"\n❓ 4. ALL WORDS CONTAINING A QUESTION MARK (?) [Unique count: {len(set(question_words))}]")
print("-" * 80)
question_sample = [word for word, count in Counter(question_words).most_common(60)]
print(question_sample)
print("=" * 80)

🔬 MARATHON COHORT: EXTRACTING PUNCTUATED WORDS ACROSS ALL 5,000 ROWS
🎉 EXTRACTION COMPLETE! COLLATING UNIQUE COHORTS...

💬 1. ALL WORDS CONTAINING A PERIOD (.) [Unique count: 8917]
--------------------------------------------------------------------------------
['.', 'it.', 'pain.', 'ago.', 'now.', 'days.', 'day.', 'me.', 'normal.', 'time.', 'months.', 'years.', 'back.', 'old.', 'help.', 'out.', 'week.', 'up.', 'side.', 'this.', 'area.', 'Dr.', '..', 'again.', 'weeks.', 'well.', 'away.', 'problem.', 'you.', 'night.', 'year.', 'month.', 'chest.', 'fine.', '...', 'pregnant.', 'down.', 'worse.', 'do.', 'all.', 'hours.', 'infection.', 'cancer.', 'negative.', 'that.', 'before.', 'too.', 'doctor.', 'bad.', 'surgery.', 'symptoms.', 'painful.', 'them.', 'Hi.', 'is.', 'mouth.', 'etc.', 'head.', 'also.', 'there.']

📌 2. ALL WORDS CONTAINING A COMMA (,) [Unique count: 4273]
--------------------------------------------------------------------------------
[',', 'Hi,', 'pain,', 'ago,', 'Hello,', 'it

In [19]:
import re
import pandas as pd
import string

print("================================================================================")
print("🧼 EXECUTING PRECISION MEDICAL CHARACTER CLEANING PIPELINE")
print("================================================================================")

def precision_medical_cleaner(text):
    if not isinstance(text, str):
        return ""

    # 1. Expand standard contractions natively first
    import contractions
    text = contractions.fix(text)

    # 2. Fix jammed commas & question marks by adding a space after them (e.g., "Hi,I" -> "Hi, I")
    text = re.sub(r'([,\?])([A-Za-z])', r'\1 \2', text)

    # 3. Clean trailing periods ONLY if they are NOT part of a medical abbreviation or decimal
    # This matches a dot followed by space/end of string, but protects 'Dr.', 'etc.', 'i.e.'
    protected_abbrevs = r'\b(Dr|etc|i\.e|e\.g|vs)\.'
    # Temporarily mask protected abbreviations
    text = re.sub(protected_abbrevs, r'\1_DOT_', text, flags=re.IGNORECASE)

    # Also mask numerical decimals so they don't lose their dots (e.g., 0.5 -> 0_DEC_5)
    text = re.sub(r'(\d+)\.(\d+)', r'\1_DEC_\2', text)

    # Now it is safe to split fused trailing periods like "12/13/13.I" -> "12/13/13 I"
    text = re.sub(r'\.([A-Za-z])', r' \1', text)

    # Remove all other trailing/isolated periods, commas, and question marks
    text = re.sub(r'[.,\?]+(?=\s|$)', '', text)

    # Restore our protected abbreviations and decimals
    text = text.replace('_DOT_', '.').replace('_DEC_', '.')

    # 4. Collapse accidental double spaces down to single spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Run the precision script on our 10 rows
raw_inputs_10 = df_clean['input'].head(10).astype(str).tolist()
df_clean.loc[df_clean.index[:10], 'precision_clean'] = [precision_medical_cleaner(t) for t in raw_inputs_10]

print("🎉 SUCCESS: Layer locked in! Let's scan for any remaining uncleaned words.")
print("=" * 80)

🧼 EXECUTING PRECISION MEDICAL CHARACTER CLEANING PIPELINE
🎉 SUCCESS: Layer locked in! Let's scan for any remaining uncleaned words.


In [20]:
print("================================================================================")
print("🔬 DIAGNOSTICS: WORDS RETAINING PUNCTUATION POST-CLEANING (FIRST 10 ROWS)")
print("================================================================================")

# Gather all tokens from our newly cleaned column
cleaned_samples = df_clean['precision_clean'].head(10).tolist()
target_punctuation = set(string.punctuation)

remaining_punctuated_words = []
for text in cleaned_samples:
    for word in text.split():
        # Catch any word that still has punctuation characters inside it
        if any(char in target_punctuation for char in word):
            remaining_punctuated_words.append(word)

# Keep only unique examples so it's easy to read
unique_remaining = sorted(list(set(remaining_punctuated_words)))

print(f"📊 Total unique punctuated words remaining: {len(unique_remaining)}")
print("-" * 80)
print("📋 Here are the words left over in your data:")
print(unique_remaining)
print("=" * 80)

🔬 DIAGNOSTICS: WORDS RETAINING PUNCTUATION POST-CLEANING (FIRST 10 ROWS)
📊 Total unique punctuated words remaining: 10
--------------------------------------------------------------------------------
📋 Here are the words left over in your data:
['(do', '(male)', '15-20', '5-6', 'ILD-Interstitial', 'doc!', 'help!', 'leg/surgery', 'rattly/raspy', 'triathlons)']


In [23]:
import re
import pandas as pd
import string

print("================================================================================")
print("🧼 EXECUTING PRECISION MEDICAL CHARACTER CLEANING PIPELINE")
print("================================================================================")

def precision_medical_cleaner(text):
    if not isinstance(text, str):
        return ""

    # 1. Expand standard contractions natively first
    import contractions
    text = contractions.fix(text)

    # 2. Fix jammed commas & question marks by adding a space after them (e.g., "Hi,I" -> "Hi, I")
    text = re.sub(r'([,\?])([A-Za-z])', r'\1 \2', text)

    # 3. Clean trailing periods ONLY if they are NOT part of a medical abbreviation or decimal
    # This matches a dot followed by space/end of string, but protects 'Dr.', 'etc.', 'i.e.'
    protected_abbrevs = r'\b(Dr|etc|i\.e|e\.g|vs)\.'
    # Temporarily mask protected abbreviations
    text = re.sub(protected_abbrevs, r'\1_DOT_', text, flags=re.IGNORECASE)

    # Also mask numerical decimals so they don't lose their dots (e.g., 0.5 -> 0_DEC_5)
    text = re.sub(r'(\d+)\.(\d+)', r'\1_DEC_\2', text)

    # Now it is safe to split fused trailing periods like "12/13/13.I" -> "12/13/13 I"
    text = re.sub(r'\.([A-Za-z])', r' \1', text)

    # Remove all other trailing/isolated periods, commas, and question marks
    text = re.sub(r'[.,\?]+(?=\s|$)', '', text)

    # Restore our protected abbreviations and decimals
    text = text.replace('_DOT_', '.').replace('_DEC_', '.')

    # 4. Collapse accidental double spaces down to single spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Run the precision script on our 10 rows
raw_inputs_10 = df_clean['input'].head(10).astype(str).tolist()
df_clean.loc[df_clean.index[:10], 'precision_clean'] = [precision_medical_cleaner(t) for t in raw_inputs_10]

print("🎉 SUCCESS: Layer locked in! Let's scan for any remaining uncleaned words.")
print("=" * 80)

🧼 EXECUTING PRECISION MEDICAL CHARACTER CLEANING PIPELINE
🎉 SUCCESS: Layer locked in! Let's scan for any remaining uncleaned words.


In [14]:
import contractions
import pandas as pd

print("================================================================================")
print("🧪 STEP 1: PROCESSING & PRINTING FIRST 10 ROWS INSTANTLY")
print("================================================================================")

# 1. Pull exactly 10 rows from your active df_clean DataFrame
raw_inputs_10 = df_clean['input'].head(10).astype(str).tolist()
raw_outputs_10 = df_clean['output'].head(10).astype(str).tolist()

# 2. Apply contractions fix FIRST, then lowercase immediately after
print("⏳ Expanding contractions and normalizing casing...")
clean_inputs_10 = [contractions.fix(text).lower() for text in raw_inputs_10]

# 3. Commit these clean rows back to a temporary column for safe verification
df_clean.loc[df_clean.index[:10], 'clean_input'] = clean_inputs_10

print("\n" + "="*80)
print("📊 DETAILED INSPECTION PANEL: 10-ROW PATIENT & DOCTOR MAPPING")
print("="*80)

# 4. Print out all 10 results directly
for idx in range(10):
    print(f"\n📍 DATASET ROW INDEX: {idx}")
    print("-" * 70)
    print(f"🔴 BEFORE PATIENT INPUT  :\n   \"{raw_inputs_10[idx]}\"")
    print(f"🟢 AFTER CONTRACTIONS/LC :\n   \"{df_clean['clean_input'].iloc[idx]}\"")
    print(f"🩺 TARGET DOCTOR ANSWER   :\n   \"{raw_outputs_10[idx]}\"")
    print("=" * 80)

🧪 STEP 1: PROCESSING & PRINTING FIRST 10 ROWS INSTANTLY
⏳ Expanding contractions and normalizing casing...

📊 DETAILED INSPECTION PANEL: 10-ROW PATIENT & DOCTOR MAPPING

📍 DATASET ROW INDEX: 0
----------------------------------------------------------------------
🔴 BEFORE PATIENT INPUT  :
   "I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!"
🟢 AFTER CONTRACTIONS/LC :
   "i woke up this morning feeling the whole room is spinning when i was sitting down. i went t

In [ ]:
# Ensure master data list is isolated
raw_documents = df_clean_final['input'].astype(str).tolist()

# =====================================================================
# 📋 STEP 1: PRE-CLEANING BASELINE ANALYSIS (STOPWORDS REMOVED FOR MATH ONLY)
# =====================================================================
print("=== 📋 STEP 1: PRE-CLEANING BASELINE PROFILE ===")

pre_tfidf = TfidfVectorizer(stop_words='english', max_features=10)
pre_tfidf_matrix = pre_tfidf.fit_transform(raw_documents)
df_pre_tfidf = pd.DataFrame({'Pre-Clean Word': pre_tfidf.get_feature_names_out(), 'TF-IDF Score': pre_tfidf_matrix.mean(axis=0).A1}).sort_values(by='TF-IDF Score', ascending=False).reset_index(drop=True)

pre_count = CountVectorizer(stop_words='english', ngram_range=(2, 3), max_features=10)
pre_count_matrix = pre_count.fit_transform(raw_documents)
df_pre_ngrams = pd.DataFrame({'Pre-Clean Phrase': pre_count.get_feature_names_out(), 'Frequency Count': pre_count_matrix.sum(axis=0).A1}).sort_values(by='Frequency Count', ascending=False).reset_index(drop=True)

html_pre = """
<div style="display: flex;">
    <div style="margin-right: 50px;">
        <h3>📊 Word Importance Baseline</h3>
        {}
    </div>
    <div>
        <h3>🏷️ Multi-Word Phrase Baseline</h3>
        {}
    </div>
</div>
""".format(df_pre_tfidf.to_html(index=False), df_pre_ngrams.to_html(index=False))
display_html(html_pre, raw=True)


# =====================================================================
# 🧼 STEP 2: CAREFULLY ORDERED DATA CLEANING PASS
# =====================================================================
print("\n" + "="*80 + "\n=== 🧼 STEP 2: EXECUTING CAREFULLY ORDERED PIPELINE ===")

def ultimate_medical_cleaner(text):
    if not isinstance(text, str): return ""

    # 1. Expand contractions to safeguard downstream word structures
    contractions = {
        r"can't": "cannot", r"don't": "do not", r"it's": "it is", r"i'm": "i am",
        r"doesn't": "does not", r"didn't": "did not", r"he's": "he is", r"she's": "she is"
    }
    for contraction, expansion in contractions.items():
        text = re.sub(contraction, expansion, text, flags=re.IGNORECASE)

    # 2. Add spaces after jammed conversational punctuation
    text = re.sub(r'([,;\.\!\?])([A-Za-z])', r'\1 \2', text)

    # 3. Strip greetings and sign-offs at boundaries
    text = re.sub(r'^\b(hi+|hello+|hey|dear)\b\s*\b(doc|doctor)?\b[,\s\.]*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(thank\s*you|thanks)\s*\b(doc|doctor)?\b[!\.]*$', '', text, flags=re.IGNORECASE)

    # 4. Extract and mask complex age features
    text = re.sub(r'\b\d+\s*(years?|yrs?|months?|mos?|mths?)\s*(old)?\b', '[REDACTED_AGE]', text, flags=re.IGNORECASE)

    # 5. Targeted Punctuation Swap: Replace punctuation with spaces, KEEPING hyphens (-) and brackets ([])
    text = re.sub(r'[^\w\s\-\[\]]', ' ', text)

    # 6. Compress formatting layouts into clear singular spaces
    return re.sub(r'\s+', ' ', text).strip()

# Apply to your dataset
df_clean_final['clean_input'] = df_clean_final['input'].apply(ultimate_medical_cleaner)
clean_documents = df_clean_final['clean_input'].tolist()

print("✅ Master Dataset Cleaned Successfully.")


# =====================================================================
# ✅ STEP 3: POST-CLEANING VERIFICATION ANALYSIS
# =====================================================================
print("\n" + "="*80 + "\n=== ✅ STEP 3: POST-CLEANING VERIFICATION PROFILE ===")

post_tfidf = TfidfVectorizer(stop_words='english', max_features=10, token_pattern=r'(?u)\b\w+\b|\[\w+\]')
post_tfidf_matrix = post_tfidf.fit_transform(clean_documents)
df_post_tfidf = pd.DataFrame({'Cleaned Term': post_tfidf.get_feature_names_out(), 'TF-IDF Score': post_tfidf_matrix.mean(axis=0).A1}).sort_values(by='TF-IDF Score', ascending=False).reset_index(drop=True)

post_count = CountVectorizer(stop_words='english', ngram_range=(2, 3), max_features=10, token_pattern=r'(?u)\b\w+\b|\[\w+\]')
post_count_matrix = post_count.fit_transform(clean_documents)
df_post_ngrams = pd.DataFrame({'Cleaned Phrase': post_count.get_feature_names_out(), 'Frequency Count': post_count_matrix.sum(axis=0).A1}).sort_values(by='Frequency Count', ascending=False).reset_index(drop=True)

html_post = """
<div style="display: flex;">
    <div style="margin-right: 50px;">
        <h3>📊 Cleaned Word Importance</h3>
        {}
    </div>
    <div>
        <h3>🏷️ Cleaned Multi-Word Phrases</h3>
        {}
    </div>
</div>
""".format(df_post_tfidf.to_html(index=False), df_post_ngrams.to_html(index=False))
display_html(html_post, raw=True)

display(df_clean_final[['input', 'clean_input']].head(3))

=== 📋 STEP 1: PRE-CLEANING BASELINE PROFILE ===


Pre-Clean Word,TF-IDF Score
pain,0.188483
hi,0.136012
old,0.131301
like,0.129531
doctor,0.123912
days,0.119800
years,0.119387
just,0.116030
ago,0.098343
right,0.097079



=== 🧼 STEP 2: EXECUTING CAREFULLY ORDERED PIPELINE ===
✅ Master Dataset Cleaned Successfully.

=== ✅ STEP 3: POST-CLEANING VERIFICATION PROFILE ===


Cleaned Term,TF-IDF Score
[redacted_age],0.230689
pain,0.179547
like,0.127874
days,0.119208
just,0.114804
doctor,0.106497
ago,0.096186
right,0.095093
time,0.095007
left,0.081199


,input,clean_input
0,I woke up this morning feeling the whole room ...,I woke up this morning feeling the whole room ...
1,My baby has been pooing 5-6 times a day for a ...,My baby has been pooing 5-6 times a day for a ...
2,"Hello, My husband is taking Oxycodone due to a...",My husband is taking Oxycodone due to a broken...


In [ ]:
import re

# Pick an excellent sample row that contains contractions, mashed punctuation, greetings, and ages
sample_raw_text = df_clean_final['input'].iloc[3]

print("================================================================================")
print("🔬 STEP-BY-STEP DATA ENGINEERING TRACE PIPELINE")
print("================================================================================\n")

print(f"🔴 [0. ORIGINAL RAW TEXT]:\n{sample_raw_text}\n")
print("-" * 80)

# --- STEP 1: CONTRACTION EXPANSION ---
text_step1 = sample_raw_text
contractions = {
    r"can't": "cannot", r"don't": "do not", r"it's": "it is", r"i'm": "i am",
    r"doesn't": "does not", r"didn't": "did not", r"he's": "he is", r"she's": "she is"
}
for contraction, expansion in contractions.items():
    text_step1 = re.sub(contraction, expansion, text_step1, flags=re.IGNORECASE)

print(f"🔄 [1. AFTER CONTRACTION EXPANSION] (e.g., don't -> do not):\n{text_step1}\n")
print("-" * 80)

# --- STEP 2: PUNCTUATION SPACING ---
text_step2 = re.sub(r'([,;\.\!\?])([A-Za-z])', r'\1 \2', text_step1)

print(f"🔄 [2. AFTER PUNCTUATION SPACING ADJUSTMENT] (e.g., Hi,I -> Hi, I):\n{text_step2}\n")
print("-" * 80)

# --- STEP 3: GREETING & SIGN-OFF REMOVAL ---
text_step3 = re.sub(r'^\b(hi+|hello+|hey|dear)\b\s*\b(doc|doctor)?\b[,\s\.]*', '', text_step2, flags=re.IGNORECASE)
text_step3 = re.sub(r'\b(thank\s*you|thanks)\s*\b(doc|doctor)?\b[!\.]*$', '', text_step3, flags=re.IGNORECASE)

print(f"🔄 [3. AFTER RULE-BASED GREETING & SIGN-OFF REMOVAL]:\n{text_step3}\n")
print("-" * 80)

# --- STEP 4: AGE ANONYMIZATION ---
text_step4 = re.sub(r'\b\d+\s*(years?|yrs?|months?|mos?|mths?)\s*(old)?\b', '[REDACTED_AGE]', text_step3, flags=re.IGNORECASE)

print(f"🔄 [4. AFTER PATTERN-BASED AGE REDACTION]:\n{text_step4}\n")
print("-" * 80)

# --- STEP 5: TARGETED PUNCTUATION FILTER ---
text_step5 = re.sub(r'[^\w\s\-\[\]]', ' ', text_step4)

print(f"🔄 [5. AFTER PUNCTUATION NORMALIZATION] (Keeps hyphens and brackets):\n{text_step5}\n")
print("-" * 80)

# --- STEP 6: WHITESPACE COMPRESSION ---
text_step6 = re.sub(r'\s+', ' ', text_step5).strip()

print(f"🟢 [6. FINAL CLEAN OUTPUT SENTENCE FOR CHATBOT]:\n{text_step6}\n")
print("================================================================================")

🔬 STEP-BY-STEP DATA ENGINEERING TRACE PIPELINE

🔴 [0. ORIGINAL RAW TEXT]:
lump under left nipple and stomach pain (male) Hi,I have recently noticed a few weeks ago a lump under my nipple, it hurts to touch and is about the size of a quarter. Also I have bern experiencing stomach pains that prevent me from eating. I immediatly feel full and have extreme pain. Please help

--------------------------------------------------------------------------------
🔄 [1. AFTER CONTRACTION EXPANSION] (e.g., don't -> do not):
lump under left nipple and stomach pain (male) Hi,I have recently noticed a few weeks ago a lump under my nipple, it hurts to touch and is about the size of a quarter. Also I have bern experiencing stomach pains that prevent me from eating. I immediatly feel full and have extreme pain. Please help

--------------------------------------------------------------------------------
🔄 [2. AFTER PUNCTUATION SPACING ADJUSTMENT] (e.g., Hi,I -> Hi, I):
lump under left nipple and stomach pa

In [ ]:
# Load execution models
nlp = spacy.load("en_core_web_md")
nlp_medical = spacy.load("en_ner_bc5cdr_md")

print("================================================================================")
print("🧬 ADVANCED/BASIC HYBRID TECHNIQUE: CLINICAL NAMED ENTITY RECOGNITION")
print("================================================================================")
patient_text = df_clean_final['clean_input'].iloc[2]
doctor_text = df_clean_final['output'].iloc[2]

doc_patient_med = nlp_medical(patient_text)
doc_doctor_med = nlp_medical(doctor_text)

print(f"\n🔴 Cleaned Patient Input:\n\"{patient_text}\"")
print("\nDetected Clinical Entities:")
if not doc_patient_med.ents:
    print("   No clinical entities detected.")
else:
    for ent in doc_patient_med.ents:
        print(f"   👉 '{ent.text}' ➔ Clinical Type: {ent.label_}")

print(f"\n🟢 Raw Doctor Output:\n\"{doctor_text[:250]}...\"")
print("\nDetected Clinical Entities:")
if not doc_doctor_med.ents:
    print("   No clinical entities detected.")
else:
    for ent in doc_doctor_med.ents:
        print(f"   👉 '{ent.text}' ➔ Clinical Type: {ent.label_}")

print("\n" + "="*80 + "\n")

print("================================================================================")
print("🧠 BASIC TECHNIQUE: TEXT SIMILARITY (SENTENCE EMBEDDINGS & COSINE)")
print("================================================================================")
doc_patient = nlp(patient_text)
doc_doctor = nlp(doctor_text)
unrelated_doctor_text = df_clean_final['output'].iloc[0]
doc_unrelated = nlp(unrelated_doctor_text)

vec_patient = doc_patient.vector.reshape(1, -1)
vec_correct_doc = doc_doctor.vector.reshape(1, -1)
vec_unrelated_doc = doc_unrelated.vector.reshape(1, -1)

sim_correct = cosine_similarity(vec_patient, vec_correct_doc)[0][0]
sim_unrelated = cosine_similarity(vec_patient, vec_unrelated_doc)[0][0]

print(f"✅ Cosine Similarity (Patient vs. Correct Doctor Match)  : {sim_correct:.4f}")
print(f"❌ Cosine Similarity (Patient vs. Unrelated Doctor Match): {sim_unrelated:.4f}\n")

print("================================================================================")
print("🎭 BASIC TECHNIQUE: SENTIMENT ANALYSIS FOR PATIENT SATISFACTION")
print("================================================================================")
painful_patient_text = df_clean_final['clean_input'].iloc[3]
blob_analysis = TextBlob(painful_patient_text)
sentiment_score = blob_analysis.sentiment.polarity

print(f"📄 Analyzing Patient Query:\n\"{painful_patient_text}\"")
print(f"\n🎭 Calculated Sentiment Polarity Score: {sentiment_score:.4f}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject